In [ ]:
import json
import platform
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.path import Path as MplPath

warnings.filterwarnings("ignore")

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
project_root = Path.cwd().resolve()
if not (project_root / "Data").exists():
    project_root = project_root.parent

# 현재 작업 중인 교통 데이터 파일 경로
SOURCE_PATH = project_root / "Data" / "ttareung" / "data.parquet"

# 여의도 판단 기준
# - 서울 행정동 경계 GeoJSON의 여의도동 polygon
# - 따릉이 지도/실시간 대여소 목록을 2026-04-06 기준으로 직접 대조해 선정한 여의도 내부 역
YEOUIDO_BOUNDARY_URL = (
    "https://raw.githubusercontent.com/southkorea/seoul-maps/master/"
    "juso/2015/json/seoul_neighborhoods_geo.json"
)

YEOUIDO_STATION_REFERENCE = [
    {"stationId": "ST-46", "stationName": "201. 진미파라곤 앞", "lat": 37.53123856, "lon": 126.92133331},
    {"stationId": "ST-47", "stationName": "202. 국민일보 앞", "lat": 37.52881622, "lon": 126.92453003},
    {"stationId": "ST-51", "stationName": "203. 국회의사당역 3번출구 옆", "lat": 37.52805710, "lon": 126.91870117},
    {"stationId": "ST-50", "stationName": "204. 국회의사당역 5번출구 옆", "lat": 37.52816391, "lon": 126.91702271},
    {"stationId": "ST-52", "stationName": "205. 산업은행 앞", "lat": 37.52626419, "lon": 126.92050934},
    {"stationId": "ST-53", "stationName": "206. KBS 앞", "lat": 37.52466583, "lon": 126.91802216},
    {"stationId": "ST-73", "stationName": "207. 여의나루역 1번출구 앞", "lat": 37.52715683, "lon": 126.93190002},
    {"stationId": "ST-55", "stationName": "209. 유진투자증권빌딩 앞", "lat": 37.52461243, "lon": 126.92783356},
    {"stationId": "ST-59", "stationName": "213. KT앞", "lat": 37.52190781, "lon": 126.91895294},
    {"stationId": "ST-60", "stationName": "214. 금융감독원 앞", "lat": 37.52302170, "lon": 126.92083740},
    {"stationId": "ST-66", "stationName": "217. NH농협은행 앞", "lat": 37.52207947, "lon": 126.93036652},
    {"stationId": "ST-62", "stationName": "219. 롯데캐슬엠파이어 옆", "lat": 37.52069473, "lon": 126.92583466},
    {"stationId": "ST-63", "stationName": "220. 미성아파트 A동 앞", "lat": 37.51936340, "lon": 126.92604828},
    {"stationId": "ST-67", "stationName": "221. 여의도초교 앞", "lat": 37.52267456, "lon": 126.93778992},
    {"stationId": "ST-68", "stationName": "222. 시범아파트버스정류장 옆", "lat": 37.51902008, "lon": 126.93688965},
    {"stationId": "ST-69", "stationName": "223. 진주아파트상가 앞", "lat": 37.51913071, "lon": 126.93337250},
    {"stationId": "ST-70", "stationName": "224. 롯데캐슬 앞", "lat": 37.52008820, "lon": 126.93236542},
    {"stationId": "ST-71", "stationName": "225. 앙카라공원 앞", "lat": 37.51758575, "lon": 126.92961884},
    {"stationId": "ST-72", "stationName": "226. 샛강역 1번출구 앞", "lat": 37.51776505, "lon": 126.92841339},
    {"stationId": "ST-424", "stationName": "270. 증권거래소후문교차로", "lat": 37.52234268, "lon": 126.92710114},
    {"stationId": "ST-425", "stationName": "271. 샛강생태공원방문자센터 앞", "lat": 37.51896286, "lon": 126.92161560},
    {"stationId": "ST-2813", "stationName": "4563. 여의도 은하아파트", "lat": 37.51850891, "lon": 126.93448639},
    {"stationId": "ST-2814", "stationName": "4564. 63스퀘어", "lat": 37.52070999, "lon": 126.93965912},
    {"stationId": "ST-2899", "stationName": "4589. KRX한국거래소(1)", "lat": 37.52338028, "lon": 126.92759705},
    {"stationId": "ST-3061", "stationName": "5853. 여의도역2번출구 앞", "lat": 37.52243423, "lon": 126.92271423},
    {"stationId": "ST-3239", "stationName": "5870. LG트윈타워 앞", "lat": 37.52833176, "lon": 126.93017578},
    {"stationId": "ST-3315", "stationName": "5878. 우리은행 앞", "lat": 37.52349091, "lon": 126.92162323},
    {"stationId": "ST-3375", "stationName": "5888.여의도성당", "lat": 37.52164459, "lon": 126.93920898},
    {"stationId": "ST-3397", "stationName": "5893.한국산업은행", "lat": 37.52814484, "lon": 126.92175293},
]

yeouido_station_ref = pd.DataFrame(YEOUIDO_STATION_REFERENCE)
display(yeouido_station_ref.head())
SOURCE_PATH

In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        last_error = None
        for encoding in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
            try:
                return pd.read_csv(path, encoding=encoding)
            except Exception as error:
                last_error = error
        raise last_error
    raise ValueError(f"지원하지 않는 파일 형식입니다: {path.suffix}")


def load_yeouido_polygon() -> list:
    local_candidates = [
        Path("/tmp/seoul_neighborhoods_geo.json"),
        project_root / "Data" / "reference" / "seoul_neighborhoods_geo.json",
    ]

    geojson = None
    for candidate in local_candidates:
        if candidate.exists():
            geojson = json.loads(candidate.read_text())
            break

    if geojson is None:
        with urllib.request.urlopen(YEOUIDO_BOUNDARY_URL) as response:
            geojson = json.load(response)

    yeouido_feature = next(
        feature
        for feature in geojson["features"]
        if feature["properties"]["EMD_KOR_NM"] == "여의도동"
    )
    return yeouido_feature["geometry"]["coordinates"][0]


def find_coordinate_columns(df: pd.DataFrame) -> tuple[str | None, str | None]:
    candidate_pairs = [
        ("위도", "경도"),
        ("latitude", "longitude"),
        ("Latitude", "Longitude"),
        ("lat", "lon"),
        ("lat", "lng"),
        ("LAT", "LON"),
        ("LAT", "LNG"),
        ("stationLatitude", "stationLongitude"),
        ("station_latitude", "station_longitude"),
    ]

    for lat_col, lon_col in candidate_pairs:
        if lat_col in df.columns and lon_col in df.columns:
            return lat_col, lon_col

    return None, None


def collect_existing_columns(df: pd.DataFrame, candidates: list[str]) -> list[str]:
    return [column for column in candidates if column in df.columns]

In [ ]:
source_df = load_table(SOURCE_PATH)
yeouido_polygon = MplPath(load_yeouido_polygon())

yeouido_station_ids = set(yeouido_station_ref["stationId"])
yeouido_station_names = set(yeouido_station_ref["stationName"])

id_columns = collect_existing_columns(
    source_df,
    [
        "시작_대여소_ID",
        "종료_대여소_ID",
        "대여소_ID",
        "stationId",
        "station_id",
        "대여소번호",
    ],
)

name_columns = collect_existing_columns(
    source_df,
    [
        "시작_대여소명",
        "종료_대여소명",
        "대여소명",
        "stationName",
        "station_name",
        "역명",
        "정류소명",
    ],
)

lat_col, lon_col = find_coordinate_columns(source_df)

coord_mask = np.zeros(len(source_df), dtype=bool)
if lat_col and lon_col:
    lat_values = pd.to_numeric(source_df[lat_col], errors="coerce")
    lon_values = pd.to_numeric(source_df[lon_col], errors="coerce")
    valid_mask = lat_values.notna() & lon_values.notna()
    points = np.column_stack([lon_values.fillna(0), lat_values.fillna(0)])
    coord_mask[valid_mask.to_numpy()] = yeouido_polygon.contains_points(
        points[valid_mask.to_numpy()],
        radius=1e-12,
    )

id_mask = np.zeros(len(source_df), dtype=bool)
for column in id_columns:
    id_mask |= source_df[column].astype(str).isin(yeouido_station_ids).to_numpy()

name_mask = np.zeros(len(source_df), dtype=bool)
for column in name_columns:
    name_mask |= source_df[column].astype(str).isin(yeouido_station_names).to_numpy()

yeouido_related_mask = coord_mask | id_mask | name_mask
yeouido_transport_df = source_df.loc[yeouido_related_mask].copy()

print(f"SOURCE_PATH: {SOURCE_PATH}")
print(f"원본 행 수: {len(source_df):,}")
print(f"여의도 관련 행 수: {len(yeouido_transport_df):,}")
print(f"좌표 컬럼: {(lat_col, lon_col) if lat_col and lon_col else '없음'}")
print(f"사용한 ID 컬럼: {id_columns if id_columns else '없음'}")
print(f"사용한 이름 컬럼: {name_columns if name_columns else '없음'}")
print(f"좌표 기준 매칭 행 수: {int(coord_mask.sum()):,}")
print(f"ID 기준 매칭 행 수: {int(id_mask.sum()):,}")
print(f"이름 기준 매칭 행 수: {int(name_mask.sum()):,}")

display(yeouido_transport_df.head())
yeouido_transport_df